Это будет телеграм-бот-контентщик для твоего канала: ты кидаешь ему в личку ссылку — либо на статью/страницу сайта, либо на пост из другого телеграм-канала — и бот сам превращает это в новый пост для твоего канала.

Когда ссылка приходит, бот понимает, что это за источник. Если это сайт, он вытаскивает заголовок, основной текст и, если возможно, картинку (например, главную или `og:image`). Если это телеграм-пост, он забирает текст и медиа (например фото). Дальше бот анализирует язык: если исходник на английском, он переводит и адаптирует текст на русский. После этого делает рерайт — то есть пишет текст заново, сохраняя смысл, но не копируя дословно, чтобы получилось “как новый пост” в формате телеграма.

Параллельно бот готовит визуал: если у источника уже есть подходящее изображение (с сайта или из поста), он сохраняет его как “исходный вариант”. Дополнительно он генерирует новое изображение по теме поста (под содержание и заголовок). Перед публикацией бот предлагает выбор, что ставить в пост: исходную картинку или сгенерированную (а при необходимости можно и без картинки, если это включим). После выбора бот публикует итоговый пост в твой канал: текст + выбранное изображение, корректно обрабатывая ограничения телеграма (например, если текст слишком длинный для подписи к фото — отправит фото и текст отдельным сообщением).

Чтобы ботом не мог пользоваться кто угодно, доступ будет ограничен по твоему user_id (whitelist). Также бот будет вести историю обработанных ссылок, чтобы не постить одно и то же дважды: если ты пришлешь тот же URL повторно, он предупредит, что это уже было. В итоге у тебя получается “конвейер”: нашел контент → кинул ссылку → получил готовый русский пост с нормальной картинкой и публикацией в канал за тебя.


Привет, мне нужен телеграм-бот для моего канала, который работает как контент-менеджер и пульт управления источниками.

Я отправляю боту ссылку — это может быть страница сайта (статья/новость/пост) или пост из другого телеграм-канала. Бот сам достает контент по ссылке, делает новый пост и публикует его у меня в канале.

Бот должен уметь:

* определять тип ссылки (сайт или Telegram-пост);
* извлекать текст и, если есть, медиа (картинки/видео), а для сайта — заголовок, основной текст и главное изображение;
* если исходный контент на английском — переводить и адаптировать на русский;
* делать рерайт/адаптацию текста (не копировать дословно, а сохранять смысл и оформлять как новый пост в Telegram-формате);
* генерировать подходящее изображение под тему поста;
* если у источника уже есть картинка (с сайта или из поста), сохранять ее как вариант;
* перед публикацией предлагать мне выбор, что использовать: исходное изображение или сгенерированное (и при необходимости — вариант “без изображения”);
* публиковать итоговый пост в мой канал (с учетом лимитов Telegram: если подпись к фото слишком длинная — фото и текст отправлять раздельно).

Отдельно: мне нужно управление API-источниками прямо через бота, потому что я буду подключать разные источники контента через их API. Я хочу в любой момент через интерфейс бота:

* выбирать, какой источник/API сейчас используется (или задавать приоритеты: сначала API, потом обычный парсинг как fallback);
* добавлять новые API-источники (создавать новый “источник” с названием, типом, настройками);
* вводить и обновлять API-ключи/токены/секреты, базовые URL, лимиты, дополнительные параметры;
* включать/выключать источники и проверять их работоспособность (тест подключения);
* настраивать правила, для каких доменов/типов ссылок какой источник должен применяться;
* хранить несколько ключей на один источник и быстро переключаться между ними.

Все ключи и токены должны храниться безопасно: не в сообщениях и не в коде, а в базе данных. Доступ к управлению и использованию бота должен быть ограничен только мной (whitelist по user_id). Бот не должен показывать ключи целиком — только маскировать. Также нужна история/лог действий (кто и когда поменял ключ, переключил источник и т.д.) и защита от дублей ссылок (чтобы одну и ту же ссылку не публиковать повторно).

**Сделаем так, шаг за шагом**

### 1. Подготовь локальный репозиторий (на ПК)
Открой PowerShell в `D:\multiagent\RL_and_PPO\tlg_bot_n` и выполни:

```powershell
git status
git add .
git commit -m "Init telegram content manager bot with docker and auto deploy"
```

Если удаленный репозиторий еще не подключен:

```powershell
git remote add origin git@github.com:USERNAME/REPO.git
git branch -M main
git push -u origin main
```

Если `origin` уже есть:

```powershell
git branch -M main
git push -u origin main
```

### 2. Подготовь VPS (один раз)
Подключись к VPS по SSH и выполни:

```bash
sudo apt update
sudo apt install -y git ca-certificates curl docker.io docker-compose-plugin
sudo systemctl enable --now docker
sudo usermod -aG docker $USER
```

Перезайди по SSH (или `newgrp docker`), потом:

```bash
mkdir -p /opt/tlg_bot_n
```

### 3. Дай VPS доступ к GitHub-репозиторию
Если репозиторий **private**, сделай так на VPS:

```bash
ssh-keygen -t ed25519 -C "vps-git-pull" -f ~/.ssh/id_ed25519_github -N ""
cat ~/.ssh/id_ed25519_github.pub
```

Скопируй вывод и добавь в GitHub:
`Repo -> Settings -> Deploy keys -> Add deploy key` (Read-only достаточно).

Создай `~/.ssh/config` на VPS:

```bash
cat > ~/.ssh/config << 'EOF'
Host github.com
  HostName github.com
  User git
  IdentityFile ~/.ssh/id_ed25519_github
  IdentitiesOnly yes
EOF
chmod 600 ~/.ssh/config
```

### 4. Первый запуск бота на VPS
На VPS:

```bash
cd /opt/tlg_bot_n
git clone git@github.com:USERNAME/REPO.git .
cp .env.example .env
nano .env
```

Заполни `.env`:
- `BOT_TOKEN`
- `CHANNEL_ID`
- `ALLOWED_USER_IDS`
- `ENCRYPTION_KEY`

Сгенерировать `ENCRYPTION_KEY` можно так:

```bash
python3 -c "from cryptography.fernet import Fernet; print(Fernet.generate_key().decode())"
```

Запусти:

```bash
docker compose up -d --build
docker compose logs -f --tail=100
```

### 5. Настрой GitHub Actions для автодеплоя
В GitHub репозитории открой:
`Settings -> Secrets and variables -> Actions -> New repository secret`

Добавь:
- `VPS_HOST` = IP/домен VPS
- `VPS_USER` = пользователь SSH (например `ubuntu`)
- `VPS_SSH_PORT` = `22`
- `VPS_DEPLOY_PATH` = `/opt/tlg_bot_n`
- `VPS_SSH_KEY` = приватный ключ для входа GitHub Actions на VPS

### 6. Создай SSH-ключ именно для GitHub Actions -> VPS
На ПК:

```powershell
ssh-keygen -t ed25519 -C "github-actions-deploy" -f .\gh_actions_vps -N ""
```

Публичный ключ (`gh_actions_vps.pub`) добавь на VPS в `~/.ssh/authorized_keys` пользователя `VPS_USER`.

Приватный ключ (`gh_actions_vps`) целиком вставь в GitHub Secret `VPS_SSH_KEY`.

### 7. Проверка автодеплоя
С ПК сделай любое изменение и пуш:

```powershell
git add .
git commit -m "Test auto deploy"
git push
```

Проверь:
`GitHub -> Actions -> Deploy Bot` (должен пройти успешно).

На VPS проверь обновление:

```bash
cd /opt/tlg_bot_n
docker compose logs -f --tail=100
```

### 8. Как дальше дорабатывать
Обычный цикл:
1. Меняешь код на ПК.
2. `git commit && git push`.
3. GitHub Actions сам обновляет VPS.

Если хочешь, следующим сообщением дам короткий чеклист “что нажимать в GitHub UI” со скрин-ориентиром по разделам (без технической воды).

git clone https://github.com/3009james/tlg_bot_news.git .



**Основные команды на VPS**

Рабочая папка:
```bash
cd /opt/tlg_bot_n
```

**Запустить бота**
Если бот еще не запущен или после изменений:
```bash
docker compose up -d --build
```

**Остановить бота**
```bash
docker compose down
```

**Перезапустить бота**
```bash
docker compose restart
```

Если менялся код или `.env`:
```bash
docker compose up -d --build
```

**Посмотреть логи**
Последние логи:
```bash
docker compose logs --tail=100
```

Логи в реальном времени:
```bash
docker compose logs -f --tail=100
```

**Проверить, запущен ли бот**
```bash
docker ps
```

**Посмотреть все контейнеры, даже остановленные**
```bash
docker ps -a
```

**Проверить статус контейнера проекта**
```bash
docker compose ps
```

**Зайти внутрь контейнера**
```bash
docker compose exec bot bash
```

Если `bash` нет:
```bash
docker compose exec bot sh
```

**Посмотреть файл `.env`**
```bash
cat .env
```

Осторожно: там секреты.

**Редактировать `.env`**
```bash
nano .env
```

После изменения `.env`:
```bash
docker compose up -d --build
```

**Посмотреть файлы проекта**
```bash
ls -la
```

**Обновить проект вручную из GitHub**
```bash
git pull origin main
docker compose up -d --build
```

**Остановить и удалить контейнеры проекта**
```bash
docker compose down
```

**Остановить и удалить контейнеры + локальные образы проекта**
```bash
docker compose down --rmi local
```

**Посмотреть использование ресурсов контейнерами**
```bash
docker stats
```

**Посмотреть имя контейнера**
У тебя сейчас контейнер называется:
```bash
tlg_bot_n
```

Можно смотреть логи и так:
```bash
docker logs -f --tail=100 tlg_bot_n
```

**Полезный минимальный набор**
Обычно хватит этих 5 команд:

```bash
cd /opt/tlg_bot_n
docker compose up -d --build
docker compose down
docker compose restart
docker compose logs -f --tail=100
docker compose ps
```

Если хочешь, следующим сообщением дам тебе готовую мини-шпаргалку в 10 строк, которую можно просто сохранить себе как памятку.